# TMDB MODELLING

## 🧩 Introduction
In this notebook, we focus on building and evaluating machine learning models to **predict movie revenue** using data collected from the **TMDB API**.  
After completing **data cleaning** and **exploratory data analysis (EDA)** in previous stages, we now aim to develop predictive models that can estimate a movie’s potential box office performance based on features such as **budget**, **popularity**, **runtime**, **genre**, and **language**.

## 🎯 Objective
The main objective of this notebook is to **train, evaluate, and compare multiple regression models** to identify which algorithm best predicts movie revenue.  
The best-performing model will later be integrated into a **machine learning pipeline** for streamlined preprocessing, tuning, and deployment.

## ⚙️ Key Tasks
1. Load the preprocessed dataset (`encoded_movies.csv`).  
2. Split the data into **training** and **testing** sets.  
3. Build and train four regression models:  
   - Linear Regression  
   - Decision Tree Regressor  
   - Random Forest Regressor  
   - XGBoost Regressor  
4. Evaluate model performance using:
   - **R² (Coefficient of Determination)**
   - **MAE (Mean Absolute Error)**
   - **RMSE (Root Mean Squared Error)**
5. Compare all models and select the best performer for pipeline integration.

## ✅ Expected Outcome
By the end of this notebook, we will have:
- A clear understanding of which model performs best for revenue prediction.  
- Quantitative performance metrics (R², MAE, RMSE) for each model.  
- A chosen model ready for **pipeline development and optimization** in the next stage.


In [1]:
#lets import the libraries
import pandas as pd
import numpy as np

In [2]:
#lets load our dataset
df=pd.read_csv('C:/Users/Win/TMDB project/data/encoded_movies.csv')
df.head()


,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western
0,26000000,92691,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,0,0,0,0,0,1,0,0
1,20000000,643612593,156,261.8345,7.793,474,0.30,11,1,0,...,1,0,0,0,0,0,0,1,0,0
2,17647000,3247000,147,271.6580,2.100,8,1.06,12,1,0,...,0,0,0,0,0,0,0,1,0,0
3,55000000,482377782,135,238.2160,7.000,1080,0.17,4,0,0,...,0,0,1,0,0,0,0,0,0,0
4,8500000,11577352,98,150.7366,5.850,70,0.11,4,0,0,...,0,0,1,0,0,0,0,1,0,0


## FEATURE ENGINEERING

In [3]:
#we create a new column budget_popularrity_ratio
df['budget_popularity_ratio']=df['budget']/df['popularity']
df.head(2)

,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio
0,26000000,92691,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,0,0,0,0,1,0,0,88414.082595
1,20000000,643612593,156,261.8345,7.793,474,0.30,11,1,0,...,0,0,0,0,0,0,1,0,0,76384.128142


In [4]:
#we log transform budget and revenue columns to handle skewness
df['revenue']=np.log1p(df['revenue'])
df['budget']=np.log1p(df['budget'])
df.head()

,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio
0,17.073607,11.437037,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,0,0,0,0,1,0,0,88414.082595
1,16.811243,20.282608,156,261.8345,7.793,474,0.30,11,1,0,...,0,0,0,0,0,0,1,0,0,76384.128142
2,16.686076,14.993242,147,271.6580,2.100,8,1.06,12,1,0,...,0,0,0,0,0,0,1,0,0,64960.354563
3,17.822844,19.994238,135,238.2160,7.000,1080,0.17,4,0,0,...,0,1,0,0,0,0,0,0,0,230882.896195
4,15.955577,16.264561,98,150.7366,5.850,70,0.11,4,0,0,...,0,1,0,0,0,0,1,0,0,56389.755375


In [5]:
#lets now bin runtime into categorical variables
# Define bins and labels
bins = [0, 90, 120, 150, df['runtime'].max()]
labels = ['Short', 'Medium', 'Long', 'Very Long']

# Create a new categorical column
df['runtime_bin'] = pd.cut(df['runtime'], bins=bins, labels=labels, include_lowest=True)
df.head()


,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio,runtime_bin
0,17.073607,11.437037,110,294.0708,6.686,35,0.16,4,1,1,...,0,0,0,0,0,1,0,0,88414.082595,Medium
1,16.811243,20.282608,156,261.8345,7.793,474,0.30,11,1,0,...,0,0,0,0,0,1,0,0,76384.128142,Very Long
2,16.686076,14.993242,147,271.6580,2.100,8,1.06,12,1,0,...,0,0,0,0,0,1,0,0,64960.354563,Long
3,17.822844,19.994238,135,238.2160,7.000,1080,0.17,4,0,0,...,1,0,0,0,0,0,0,0,230882.896195,Long
4,15.955577,16.264561,98,150.7366,5.850,70,0.11,4,0,0,...,1,0,0,0,0,1,0,0,56389.755375,Medium


In [6]:
# lets now ecode the runtime bin , using onehot encoding
# Convert boolean dummies to integers
df = pd.get_dummies(df, columns=['runtime_bin'], prefix='runtime', drop_first=True).astype(int)
df.head(2)


,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Mystery,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio,runtime_Medium,runtime_Long,runtime_Very Long
0,17,11,110,294,6,35,0,4,1,1,...,0,0,0,1,0,0,88414,1,0,0
1,16,20,156,261,7,474,0,11,1,0,...,0,0,0,1,0,0,76384,0,0,1


In [7]:
# we check whether we have categorical columns and see the correlation
df.corr()

,budget,revenue,runtime,popularity,vote_average,vote_count,movie_age,original_language_encoded,Action,Adventure,...,Mystery,Romance,Science Fiction,Thriller,War,Western,budget_popularity_ratio,runtime_Medium,runtime_Long,runtime_Very Long
budget,1.000000,0.568362,0.270746,0.081285,-0.081326,0.337292,-0.374192,-0.185285,0.310755,0.362853,...,-0.053254,-0.056641,0.195988,-0.083763,0.005136,-0.035128,0.717386,-0.102417,0.185764,0.102677
revenue,0.568362,1.000000,0.205044,0.042049,0.099679,0.498152,-0.040222,-0.184172,0.135237,0.279006,...,-0.028546,-0.022179,0.129487,-0.122144,-0.023416,-0.035254,0.422017,-0.112975,0.164511,0.078331
runtime,0.270746,0.205044,1.000000,0.106363,0.297767,0.282979,-0.038609,0.131000,0.185752,0.065805,...,0.006473,0.028527,0.068643,0.010936,0.158880,0.072289,0.191257,-0.454179,0.475269,0.652015
popularity,0.081285,0.042049,0.106363,1.000000,0.057828,0.067258,-0.168239,0.049832,0.056233,0.057317,...,-0.028458,-0.039754,0.034291,0.046621,-0.013699,-0.016427,-0.107941,-0.062844,0.042172,0.090693
vote_average,-0.081326,0.099679,0.297767,0.057828,1.000000,0.384325,0.142062,0.103310,-0.118361,-0.020953,...,0.000815,0.018937,-0.019485,-0.087927,0.092548,0.055219,-0.173825,-0.169884,0.177533,0.180051
vote_count,0.337292,0.498152,0.282979,0.067258,0.384325,1.000000,0.004523,-0.142119,0.125561,0.206415,...,-0.022014,-0.066846,0.227769,-0.090993,0.003970,-0.002231,0.239771,-0.167727,0.190930,0.155701
movie_age,-0.374192,-0.040222,-0.038609,-0.168239,0.142062,0.004523,1.000000,-0.095935,-0.119051,-0.040205,...,0.008525,0.072235,-0.065499,-0.033940,0.065819,0.073823,-0.228936,-0.052126,-0.044510,0.024241
original_language_encoded,-0.185285,-0.184172,0.131000,0.049832,0.103310,-0.142119,-0.095935,1.000000,0.087547,0.011204,...,-0.025741,-0.016117,-0.024650,-0.033820,0.026070,-0.006805,-0.138534,-0.063715,0.027112,0.128820
Action,0.310755,0.135237,0.185752,0.056233,-0.118361,0.125561,-0.119051,0.087547,1.000000,0.300408,...,-0.139404,-0.232091,0.288613,0.178651,0.060095,-0.012378,0.311470,-0.078334,0.167720,0.041006
Adventure,0.362853,0.279006,0.065805,0.057317,-0.020953,0.206415,-0.040205,0.011204,0.300408,1.000000,...,-0.142268,-0.161565,0.207683,-0.182698,-0.039655,0.027180,0.406314,-0.105954,0.100948,0.015758


- **NOW HAVING ENGINEERED ALL THE FEATURURES WE NEED WE MOVE TO MODELLING**

## MODELLING

In [8]:
#WE now split our data into training and testing
from sklearn.model_selection import train_test_split

In [9]:
#we separate the features fro the target
X=df.drop('revenue',axis=1)
y=df['revenue']

In [10]:
#we split the dataset into training and testing sets
#We Split the data into an 80% training set and a 20% testing set.
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 1. Linear Regression

In [11]:
#imports
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [12]:
#we initialize the model
lin_reg=LinearRegression()

In [13]:
#we train our model
lin_reg.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [14]:
# lets make the predictions 
y_pred=lin_reg.predict(X_test)

In [15]:
#model evaluations
r2=r2_score(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
rmse=np.sqrt(mean_squared_error(y_test,y_pred))

print(f"R^2:{r2:.4f}")
print(f"MAE:{mae:.4f}")
print(f"RMSE:{rmse:.4f}")

R^2:0.4122
MAE:0.9524
RMSE:1.5472


- **We Reverse the log-transformation (np.expm1) on predictions and test values to evaluate performance on the original dollar scale.**

In [16]:
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_original = np.expm1(y_pred)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_original)
mae = mean_absolute_error(y_test_original, y_pred_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

R²: -0.8118
MAE: 98878627.0883
RMSE: 290627437.1630


## 2. Decision Tree Regressor

In [17]:
#importing the model
from sklearn.tree import DecisionTreeRegressor

In [18]:
#we initialize our model
dt_reg=DecisionTreeRegressor(random_state=42)

In [19]:
#we train our model
dt_reg.fit(X_train,y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [20]:
#we predict using our model
y_pred_dt=dt_reg.predict(X_test)

In [21]:
#evaluation
r2=r2_score(y_test,y_pred_dt)
mae=mean_absolute_error(y_test,y_pred_dt)
rmse=np.sqrt(mean_squared_error(y_test,y_pred_dt))
print(f"R^2:{r2:.4f}")
print(f"MAE:{mae:.4f}")
print(f"RMSE:{rmse:.4f}")

R^2:0.2282
MAE:1.0859
RMSE:1.7729


- **We Reverse the log-transformation (np.expm1) on predictions and test values to evaluate performance on the original dollar scale.**

In [22]:
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_dt_original = np.expm1(y_pred_dt)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_dt_original)
mae = mean_absolute_error(y_test_original, y_pred_dt_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_dt_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

R²: 0.3805
MAE: 78862572.7125
RMSE: 169942852.4256


## 3. Random Forest Regressor

In [23]:
#importing the model
from sklearn.ensemble import RandomForestRegressor

In [24]:
#we initialize or model
rf_reg=RandomForestRegressor(random_state=42)

In [25]:
#we train our model
rf_reg.fit(X_train,y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
#we make predictions
y_pred_rf=rf_reg.predict(X_test)

In [27]:
#evaluations
r2=r2_score(y_test,y_pred_rf)
mae=mean_absolute_error(y_test,y_pred_rf)
rmse=np.sqrt(mean_squared_error(y_test,y_pred_rf))
print(f"(R^2:{r2:.4f}")
print(f"(MAE:{mae:.4f}")
print(f"(RMSE:{rmse:.4f}")

(R^2:0.4832
(MAE:0.8593
(RMSE:1.4508


- **We Reverse the log-transformation (np.expm1) on predictions and test values to evaluate performance on the original dollar scale.**

In [28]:
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_rf_original = np.expm1(y_pred_rf)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_rf_original)
mae = mean_absolute_error(y_test_original, y_pred_rf_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_rf_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

R²: 0.5212
MAE: 68464761.2344
RMSE: 149403008.2841


## 4. XGBoostregressor

In [29]:
#importing the model
from xgboost import XGBRegressor

In [30]:
#we initialize our model
xgb_model=XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [31]:
#we train our model
xgb_model.fit(X_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [32]:
#we predict
y_pred_xgb=xgb_model.predict(X_test)

In [33]:
#evaluations
r2=r2_score(y_test,y_pred_xgb)
mae=mean_absolute_error(y_test,y_pred_xgb)
rmse=np.sqrt(mean_squared_error(y_test,y_pred_xgb))
print(f"(R^2:{r2:.4f}")
print(f"(MAE:{mae:.4f}")
print(f"(RMSE:{rmse:.4f}")

(R^2:0.4851
(MAE:0.8472
(RMSE:1.4481


- **We Reverse the log-transformation (np.expm1) on predictions and test values to evaluate performance on the original dollar scale.**

In [34]:
#evaluation
# Reverse the log transformation
y_test_original = np.expm1(y_test)
y_pred_xgb_original = np.expm1(y_pred_xgb)

# Recalculate metrics on the original scale
r2 = r2_score(y_test_original, y_pred_xgb_original)
mae = mean_absolute_error(y_test_original, y_pred_xgb_original)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_xgb_original))

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")


R²: 0.5378
MAE: 70072567.6513
RMSE: 146790839.4670


# Model Comparison and Insights
- **This section is for comparing all model results and deriving conclusions.**

In [35]:
#lets see the original revenue mean
original_revenue=np.expm1(df['revenue'])
np.mean(original_revenue)

np.float64(128210874.98363106)

- To select the best algorithm, we will compare the performance of all four models on the original (non-log-transformed) data. The key metrics are R² (how much variance the model explains) and MAE/RMSE (how large the prediction error is in dollars).

We will use the **mean revenue of \$128,210,875** to provide critical context for the error metrics.

### Performance on Original Dollar Scale

| Model | R² Score (Higher is Better) | MAE (Lower is Better) | RMSE (Lower is Better) |
| :--- | :--- | :--- | :--- |
| **XGBoost Regressor** | **0.5378** | \$70,072,567 | **\$146,790,839** |
| **Random Forest** | 0.5212 | **\$68,464,761** | \$149,403,008 |
| **Decision Tree** | 0.3805 | \$78,862,572 | \$169,942,852 |
| **Linear Regression** | -0.8118 | \$98,878,627 | \$290,627,437 |

### Analysis

The average revenue for a movie in this dataset is approximately **\$128.2 million**.

* **Linear Regression:** Performed very poorly, with a negative R² score. This confirms the model is not viable. Its average error (MAE) of **\$98.9M** is ~77% of the mean revenue, meaning its predictions are drastically inaccurate.
* **Decision Tree:** Showed significant improvement, but its average error of **\$78.9M** is still high, at ~61% of the mean.
* **Random Forest:** Performed very well, achieving the **best (lowest) MAE at \$68.5M**. This means its predictions are, on average, off by **~53%** of the mean revenue.
* **XGBoost Regressor:** The **top overall performer**. It achieved the **best R² score** (explaining ~54% of revenue variance) and the **best RMSE**. Its MAE of **\$70.1M** is slightly higher than Random Forest's (~55% of the mean).

**Key Insight:** For both Random Forest and XGBoost, the **RMSE is more than double the MAE** (and also higher than the mean revenue itself). This strongly suggests that while the *average* error is high (~53-55% of the mean), the models are making a few **very large errors** on high-revenue (blockbuster) movies, which heavily penalizes the RMSE score. This is a key area to investigate during tuning.

### Selection for Hyperparameter Tuning

Based on these results, the two clear candidates for optimization are the ensemble methods.

1.  **XGBoost Regressor** (Top performer on R² and RMSE)
2.  **Random Forest Regressor** (Top performer on MAE and very close second overall)

We will proceed with hyperparameter tuning for these two models to see if we can further improve their predictive accuracy and, ideally, reduce the impact of these large outlier errors.

# HYPERPARAMETER TUNING

## 1. RandomForest
- ## 1a. **RandomizedSearchCv**

In [36]:
# Import the necessary tools
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [37]:
#  Define the Parameter Search Space ---
# We create a dictionary of parameters to try.
# We'll use randint to sample from a range of values.
param_dist = {
    # n_estimators: Number of trees in the forest
    'n_estimators': randint(200, 1000),

    # max_depth: Maximum depth of each tree
    'max_depth': [None, 10, 20, 30, 40, 50],

    # min_samples_split: Minimum number of samples required to split an internal node
    'min_samples_split': randint(2, 11),  # Randomly sample integers from 2 to 10

    # min_samples_leaf: Minimum number of samples required to be at a leaf node
    'min_samples_leaf': randint(1, 5),   # Randomly sample integers from 1 to 4
    
    # max_features: Number of features to consider when looking for the best split
    'max_features': ['sqrt', 'log2', 1.0] # 1.0 is the same as the old 'auto'
}
 

In [38]:
#  Initialize the Base Model ---
# We use the same random_state for reproducibility.
rf = RandomForestRegressor(random_state=42)


In [39]:
#  Initialize the Randomized Search ---
# This will test 'n_iter' random combinations using 5-fold cross-validation.
rf_random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=100,         # Try 100 different random combinations
    cv=5,               # Use 5-fold cross-validation
    scoring='r2',       # We'll optimize for R² on the log-transformed data
    n_jobs=-1,          # Use all available CPU cores
    random_state=42,    # For reproducible search results
    verbose=2           # Show progress as it runs
)

In [40]:
# Fit the Search ---
# This step will take some time. It's running 100 * 5 = 500 model fits.
print("Starting Random Forest tuning...")
rf_random_search.fit(X_train, y_train)
print("Tuning finished.")

Starting Random Forest tuning...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Tuning finished.


In [41]:
#lets get the Best Results ---
print("Best R² score from tuning (on log scale):")
print(rf_random_search.best_score_)
print("\nBest parameters found:")
print(rf_random_search.best_params_)


Best R² score from tuning (on log scale):
0.5429039491329528

Best parameters found:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 905}


In [42]:
#Lets get the Best Model ---
# This is the single best model, retrained on the *entire* X_train set.
best_rf_model = rf_random_search.best_estimator_


In [43]:
#lets  evaluate on the Test Set ---
# Now we use the locked X_test for our final, unbiased evaluation.
y_pred_best_rf = best_rf_model.predict(X_test)


In [44]:
#Evaluate on ORIGINAL DOLLAR SCALE ---
# We must reverse the log transformation to get meaningful dollar errors.
y_test_original = np.expm1(y_test)
y_pred_best_rf_original = np.expm1(y_pred_best_rf)

In [45]:
# Recalculate metrics on the original scale
r2_rf_tuned = r2_score(y_test_original, y_pred_best_rf_original)
mae_rf_tuned = mean_absolute_error(y_test_original, y_pred_best_rf_original)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test_original, y_pred_best_rf_original))

print("\n Tuned Random Forest Performance (Original Dollars) ")
print(f"R²: {r2_rf_tuned:.4f}")
print(f"MAE: {mae_rf_tuned:.4f}")
print(f"RMSE: {rmse_rf_tuned:.4f}")


 Tuned Random Forest Performance (Original Dollars) 
R²: 0.4740
MAE: 70886283.0897
RMSE: 156588599.1651


- ## 1b.**GridSearchCv**

In [46]:
# Import the necessary tool
from sklearn.model_selection import GridSearchCV

In [47]:
#we define the Focused Parameter Grid 
# Our baseline (default) was:
# n_estimators=100, max_depth=None, max_features=1.0

param_grid = {
    # Number of trees
    'n_estimators': [100, 200, 300], 
    
    # Max depth of each tree
    'max_depth': [None, 20, 30],
    
    # Number of features to consider
    'max_features': [1.0, 'sqrt'] 
}

In [48]:
# we initialize the Base Model 
grid_rf = RandomForestRegressor(random_state=42)


In [49]:
#we initialize the Grid Search 
# This will try EVERY combination in the grid: 3 * 3 * 2 = 18 combinations.
# Total fits = 18 * 5 (cv) = 90 fits. This is very manageable.
rf_grid_search = GridSearchCV(
    estimator=grid_rf,
    param_grid=param_grid,
    cv=5,               # Use 5-fold cross-validation
    scoring='r2',       # Optimize for R² on the log-transformed data
    n_jobs=-1,          # Use all available CPU cores
    verbose=2           # Show progress
)


In [50]:
#lets fit the Search 
print("Starting Random Forest Grid Search...")
rf_grid_search.fit(X_train, y_train)
print("Grid Search finished.")

Starting Random Forest Grid Search...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Grid Search finished.


In [51]:
#lets get the Best Results ---
print("Best R² score from tuning (on log scale):")
print(rf_grid_search.best_score_)
print("\nBest parameters found:")
print(rf_grid_search.best_params_)


Best R² score from tuning (on log scale):
0.5401913173398027

Best parameters found:
{'max_depth': 20, 'max_features': 'sqrt', 'n_estimators': 300}


In [52]:
#Lets get the Best Model ---
best_rf_grid_model = rf_grid_search.best_estimator_


In [53]:
#lets evaluate on the Test Set 
y_pred_best_rf_grid = best_rf_grid_model.predict(X_test)


In [54]:
# lets evaluate on ORIGINAL DOLLAR SCALE 
y_test_original = np.expm1(y_test)
y_pred_best_rf_grid_original = np.expm1(y_pred_best_rf_grid)


In [55]:
# Recalculate metrics on the original scale
r2_rf_grid = r2_score(y_test_original, y_pred_best_rf_grid_original)
mae_rf_grid = mean_absolute_error(y_test_original, y_pred_best_rf_grid_original)
rmse_rf_grid = np.sqrt(mean_squared_error(y_test_original, y_pred_best_rf_grid_original))


In [56]:
print("\nTuned Random Forest (Grid Search) Performance (Original Dollars) ")
print(f"R²: {r2_rf_grid:.4f}")
print(f"MAE: {mae_rf_grid:.4f}")
print(f"RMSE: {rmse_rf_grid:.4f}")


Tuned Random Forest (Grid Search) Performance (Original Dollars) 
R²: 0.4887
MAE: 70059528.1084
RMSE: 154394718.1153


## 2. XGBoostRegressor

- ## 2.a **Randomized search**

In [70]:
# Import the necessary tools
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [71]:
# we define the Parameter Search Space 
# We define a dictionary of parameters for XGBoost.
# 'uniform' samples floats, 'randint' samples integers.
param_dist = {
    # n_estimators: Number of boosting rounds (trees)
    'n_estimators': randint(100, 1000),

    # max_depth: Max depth of a tree
    'max_depth': randint(3, 10), # Default is 3, let's try a bit deeper

    # learning_rate: Step size shrinkage to prevent overfitting
    'learning_rate': uniform(loc=0.01, scale=0.29), # Range from 0.01 to 0.30

    # subsample: Fraction of observations to be randomly sampled for each tree
    'subsample': uniform(loc=0.7, scale=0.3), # Range from 0.7 to 1.0
    
    # colsample_bytree: Fraction of features to be randomly sampled for each tree
    'colsample_bytree': uniform(loc=0.7, scale=0.3) # Range from 0.7 to 1.0
}


In [72]:
#we nitialize the Base Model 
# We create a new, untrained instance.
xgb = XGBRegressor(random_state=42)

In [73]:
#we initialize the Randomized Search 
# We'll try 100 random combinations using 5-fold cross-validation.
xgb_random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=100,         # Try 100 different random combinations
    cv=5,               # Use 5-fold cross-validation
    scoring='r2',       # Optimize for R² on the log-transformed data
    n_jobs=-1,          # Use all available CPU cores
    random_state=42,    # For reproducible search results
    verbose=2           # Show progress
)

In [74]:
# we now fit the Search 
# This will take some time to run.
print("Starting XGBoost tuning...")
xgb_random_search.fit(X_train, y_train)
print("Tuning finished.")

Starting XGBoost tuning...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Tuning finished.


In [75]:
# lets get the Best Results
print("Best R² score from tuning (on log scale):")
print(xgb_random_search.best_score_)
print("\nBest parameters found:")
print(xgb_random_search.best_params_)


Best R² score from tuning (on log scale):
0.544981837272644

Best parameters found:
{'colsample_bytree': np.float64(0.8834959481464842), 'learning_rate': np.float64(0.012049228513718048), 'max_depth': 3, 'n_estimators': 660, 'subsample': np.float64(0.8574323980775167)}


In [76]:
# lets get the Best Model 
# This is the single best model found by the search.
best_xgb_model = xgb_random_search.best_estimator_


In [77]:
# lets evaluate on the Test Set 
y_pred_best_xgb = best_xgb_model.predict(X_test)


In [78]:
#Evaluation on ORIGINAL DOLLAR SCALE 
y_test_original = np.expm1(y_test)
y_pred_best_xgb_original = np.expm1(y_pred_best_xgb)


In [79]:
# Recalculating metrics on the original scale
r2_xgb_tuned = r2_score(y_test_original, y_pred_best_xgb_original)
mae_xgb_tuned = mean_absolute_error(y_test_original, y_pred_best_xgb_original)
rmse_xgb_tuned = np.sqrt(mean_squared_error(y_test_original, y_pred_best_xgb_original))


In [80]:
print("\nTuned XGBoost Performance (Original Dollars)")
print(f"R²: {r2_xgb_tuned:.4f}")
print(f"MAE: {mae_xgb_tuned:.4f}")
print(f"RMSE: {rmse_xgb_tuned:.4f}")


Tuned XGBoost Performance (Original Dollars)
R²: 0.5505
MAE: 68520713.2838
RMSE: 144760660.3820


- ##  2.b **GridSearchCv**

In [81]:
# we define the FOCUSED Parameter Grid 
# Centered around our best RandomizedSearchCV results:
# 'colsample_bytree': 0.88
# 'learning_rate': 0.012
# 'max_depth': 3
# 'n_estimators': 660
# 'subsample': 0.86

param_grid = {
    'colsample_bytree': [0.85, 0.88, 0.90],  
    'learning_rate':    [0.01, 0.012, 0.015],
    'max_depth':        [3, 4],             # Your best was 3, let's test 3 & 4
    'n_estimators':     [600, 660, 700],
    'subsample':        [0.80, 0.85, 0.90]
}


In [82]:
# we initialize the Base Model 
xgb = XGBRegressor(random_state=42)

In [83]:
# we initialize the Grid Search 
# Total combinations: 3 * 3 * 2 * 3 * 3 = 162
# Total fits = 162 * 5 (cv) = 810 fits.
xgb_grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,               # Use 5-fold cross-validation
    scoring='r2',       # Optimize for R² on the log-transformed data
    n_jobs=-1,          # Use all available CPU cores
    verbose=2           # Show progress
)

In [84]:
# we now fit the Search 
print("Starting XGBoost Grid Search...")
xgb_grid_search.fit(X_train, y_train)
print("Grid Search finished.")

Starting XGBoost Grid Search...
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Grid Search finished.


In [85]:
#lets get the Best Results 
print("Best R² score from tuning (on log scale):")
print(xgb_grid_search.best_score_)
print("\nBest parameters found:")
print(xgb_grid_search.best_params_)

Best R² score from tuning (on log scale):
0.5490889072418212

Best parameters found:
{'colsample_bytree': 0.85, 'learning_rate': 0.015, 'max_depth': 3, 'n_estimators': 700, 'subsample': 0.9}


In [86]:
#lets get the Best Model 
final_model = xgb_grid_search.best_estimator_

In [87]:
# Evaluation on the Test Set 
y_pred_final = final_model.predict(X_test)

In [88]:
# Evaluation on ORIGINAL DOLLAR SCALE 
y_test_original = np.expm1(y_test)
y_pred_final_original = np.expm1(y_pred_final)


In [89]:
# Recalculate metrics on the original scale
r2_final = r2_score(y_test_original, y_pred_final_original)
mae_final = mean_absolute_error(y_test_original, y_pred_final_original)
rmse_final = np.sqrt(mean_squared_error(y_test_original, y_pred_final_original))

print("\n--- Final Tuned XGBoost (Grid Search) Performance (Original Dollars) ---")
print(f"R²: {r2_final:.4f}")
print(f"MAE: {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")


--- Final Tuned XGBoost (Grid Search) Performance (Original Dollars) ---
R²: 0.5744
MAE: 67412205.8764
RMSE: 140863770.3958


##  Final Model Comparison & Conclusion

We have now built, evaluated, and tuned our models. The final step is to compare our best-performing models to select a single champion for our pipeline.

### Final Model Performance

This table shows the performance of the **best version** of each model we tested, evaluated on the original dollar scale.

| Model Version | R² Score (Higher is Better) | MAE (Lower is Better) | RMSE (Lower is Better) |
| :--- | :--- | :--- | :--- |
| **Tuned XGBoost (Grid Search)** | **0.5744** | **\$67,412,206** | **\$140,863,770** |
| Tuned XGBoost (Random Search) | 0.5505 | \$68,520,713 | \$144,760,660 |
| Default XGBoost | 0.5378 | \$70,072,567 | \$146,790,839 |
| Default Random Forest | 0.5212 | \$68,464,761 | \$149,403,008 |
| Tuned Random Forest (Grid) | 0.4887 | \$70,059,528 | \$154,394,718 |
| Tuned Random Forest (Random) | 0.4740 | \$70,886,283 | \$156,588,599 |
| Default Decision Tree | 0.3805 | \$78,862,572 | \$169,942,852 |
| Default Linear Regression | -0.8118 | \$98,878,627 | \$290,627,437 |

---

##  Conclusion & Insights

### 🏆 The Champion Model

The **Tuned XGBoost Regressor (from `GridSearchCV`)** is the clear and definitive winner. It outperformed all other models across all three evaluation metrics:

1.  **Highest R² (0.5744):** It explains approximately **57.4%** of the variance in movie revenue, the most of any model.
2.  **Lowest MAE (\$67.4M):** Its predictions are the most accurate on average.
3.  **Lowest RMSE (\$140.9M):** It is the most effective at avoiding very large, costly prediction errors.

### 💡 Key Insights

* **Tuning was Critical:** Hyperparameter tuning was highly effective. Our baseline XGBoost model (R²: 0.538) saw a significant performance boost from randomized search (R²: 0.551) and another solid jump from focused grid search (R²: 0.574).
* **Contextualizing Error:** Our final model's average error is **\$67.4 million**. Compared to the dataset's mean revenue of **\$128.2 million**, this means our model's predictions are, on average, off by about **52.6%**. This highlights the inherent difficulty and high variance in predicting box office success, even with good data.
* **Model Selection Matters:** The ensemble methods (XGBoost, Random Forest) were dramatically better than the simpler Linear Regression and Decision Tree models, confirming that the relationships in the data are complex and non-linear.
* **Tuning Isn't Always Better:** For Random Forest, the default parameters (R²: 0.521) were more robust than any of our tuned versions, which appeared to overfit the training data and performed worse on the test set.

**Final Decision:** The model `final_model` (the `best_estimator_` from our `xgb_grid_search`) is our champion model and will be the one we productionalize.

---

## Next Steps: Pipeline Development

The modeling and selection phase is complete. Our next objective is to build a robust, reproducible production **pipeline**.

This will involve:
1.  **Analyzing Feature Importances** from our `final_model` to understand *what* drives its predictions.
2.  **Creating a `ColumnTransformer`** to perform all of our feature engineering steps (log-transforming, binning, encoding) in one object.
3.  **Building a `scikit-learn` Pipeline** that combines the `ColumnTransformer` with our `final_model`.
4.  **Training this single pipeline** on the full dataset and **saving it to a file** (e.g., `final_model.pkl`) so it can be easily loaded in another script or application for predictions.